# DZ11 — Крос-таблиці та ймовірності

## Вправа 1. Крос-таблиця за шкалою ECTS

In [1]:
import pandas as pd

In [2]:
# Зчитуємо таблицю оцінок із CSV у DataFrame `df`
# Файл: scores.csv (очікуємо колонки: math, economics, ukrainian з балами 0..100)
df = pd.read_csv('scores.csv')
# У Jupyter/Colab останній вираз у комірці виводиться як таблиця.
# Тому просто написання `df` нижче відобразить увесь DataFrame (переглянемо дані після зчитування).
print(df.head())

   id  math  economics  ukrainian
0   1    73         20         19
1   2    46         36         84
2   3    78         48         40
3   4    56         91         37
4   5    50         10         88


In [3]:
# Перетворюємо бали у літерні оцінки ECTS для кожного предмета (math, economics, ukrainian)
for i in ['math', 'economics', 'ukrainian']:
    # формуємо назву нової колонки з мітками ECTS
    col_mark = f"{i}_mark"
    # квантуємо числовий бал df[i] у категорії ECTS за заданими межами
    df[col_mark] = pd.cut(
        df[i],
        bins=[0, 34, 59, 63, 73, 81, 89, 100],        # межі інтервалів (праві межі включені)
        labels=['F', 'FX', 'E', 'D', 'C', 'B', 'A'],  # відповідні літери ECTS
        include_lowest=True                            # включає 0 у перший інтервал
    )

# (опціонально) швидко глянути результат для перших рядків
df[['math','math_mark','economics','economics_mark','ukrainian','ukrainian_mark']].head()

,math,math_mark,economics,economics_mark,ukrainian,ukrainian_mark
0,73,D,20,F,19,F
1,46,FX,36,FX,84,B
2,78,C,48,FX,40,FX
3,56,FX,91,A,37,FX
4,50,FX,10,F,88,B


In [4]:
# Переводимо дані з «широкого» формату у «довгий» для зручної агрегації (крос-таблиці, групування тощо).
# Було: по кожному студенту (рядок) три окремі колонки з літерними оцінками:
#       math_mark, economics_mark, ukrainian_mark
# Стає: по кожному студенту кілька рядків — по одному на кожен предмет.

updated_df = df.melt(
    id_vars="id",                                         # фіксуємо ідентифікатор студента
    value_vars=['math_mark', 'economics_mark', 'ukrainian_mark'],  # перетворюємо їх на дві колонки: назва ознаки (subject) і її значення (mark)
    var_name="subject",                                   # назва нової колонки з назвою предмета
    value_name="mark"                                     # назва нової колонки з літерною оцінкою (F/FX/E/D/C/B/A)
)

# Результат:
# - Кожен студент матиме до 3 рядків (за кількістю предметів у value_vars).
# - 'subject' містить назву предмета (наприклад, 'math_mark', 'economics_mark', 'ukrainian_mark').
# - 'mark' містить літерну оцінку ECTS для відповідного предмета.
# Такий «довгий» формат зручно передавати в pd.crosstab / groupby для підрахунків частот.
updated_df  # у ноутбуку виведеться як таблиця для швидкої перевірки

,id,subject,mark
0,1,math_mark,D
1,2,math_mark,FX
2,3,math_mark,C
3,4,math_mark,FX
4,5,math_mark,FX
...,...,...,...
67,20,ukrainian_mark,A
68,21,ukrainian_mark,F
69,22,ukrainian_mark,C
70,23,ukrainian_mark,FX


In [5]:
# Очищаємо назви предметів у колонці 'subject', прибираючи суфікс "_mark".
# Було після melt: 'math_mark', 'economics_mark', 'ukrainian_mark'
# Стає           : 'math', 'economics', 'ukrainian'
# .str.replace("_mark", "") — векторизована операція для всіх рядків у Series.
# Тут це безпечно, бо значення 'subject' мають фіксовану форму "*_mark".
# Зауваження: якщо десь у рядку зустрінеться "_mark" усередині назви,
# воно теж буде видалене. Для жорсткого видалення саме суфікса можна було б
# використати regex: .str.replace(r"_mark$", "", regex=True) або .str.removesuffix("_mark")

updated_df['subject'] = updated_df['subject'].str.replace("_mark", "")

# Швидка перевірка результату:
# print(updated_df['subject'].unique())  # очікуємо: ['math', 'economics', 'ukrainian']

In [6]:
# Рахуємо крос-таблицю частот:
#   рядки  — літерні оцінки ECTS (mark: 'A','B','C','D','E','F','FX'),
#   стовпці — предмети (subject: 'math','economics','ukrainian').
# Кожна клітинка показує КІЛЬКІСТЬ студентів із відповідною літерою в конкретному предметі.

table = pd.crosstab(updated_df["mark"], updated_df["subject"])

# У ноутбуці просто 'table' в кінці комірки виведе таблицю для перевірки.
table

# (Опційно) якщо потрібно зафіксувати порядок рядків/стовпців, можна перевпорядкувати:
# desired_rows = ['A','B','C','D','E','F','FX']
# desired_cols = ['math','economics','ukrainian']
# table = table.reindex(index=desired_rows, columns=desired_cols, fill_value=0)


subject,economics,math,ukrainian
mark,,,
F,8,6,3
FX,10,10,11
E,0,2,0
D,1,2,0
C,2,3,4
B,0,1,4
A,3,0,2


**Підсумкова інтерпретація:**
- Усього студентів: **24** у кожному предметі (разом **72** оцінки).
- Крос-таблиця показує, **скільки студентів** отримали кожну **літеру ECTS** (`A, B, C, D, E, FX, F`) у **кожному предметі**.
- **Найпоширеніша літера загалом:** **FX = 31** (із 72 оцінок). Далі: **F = 17**, **C = 9**, **B = 5**, **A = 5**, **D = 3**, **E = 2**.
- **Кількість записів за предметами** однакова: **math = 24**, **economics = 24**, **ukrainian = 24**.

**Що це означає:**
- У групі домінують **низькі рівні** (**FX/F**), високих **A/B** небагато.
- Порівняння предметів за «верхом» і «низом»:
  - **Ukrainian:** `A+B = 6` (найбільше), `FX+F = 14` (найменше) → **кращий розподіл** серед трьох.
  - **Math:** `A+B = 1`, `FX+F = 16` → посередньо, високих майже нема.
  - **Economics:** `A+B = 3`, `FX+F = 18` → **найслабший** розподіл (багато FX/F).
- Таблиця зручна для **зведеної статистики**: швидко видно, де накопичуються **низькі оцінки** (цільові групи підтримки) і де **більше високих** (A/B).  
  
**Примітка:**  
- У нашій крос-таблиці рядки — це літери (`mark`), а стовпці — предмети (`subject`), тобто значення в клітинках — кількість студентів із даною літерою в конкретному предметі.

---

## Вправа 2. Ймовірність «рівно 4 ейси з 15 подач» (p = 0.18)

In [ ]:
# Імпортуємо біноміальний коефіцієнт C(n, k) з модуля math
from math import comb

def binom(quantity, prob, likely):  # біноміальний розподіл
    """
    Повертає P(X = likely) для X ~ Bin(n=quantity, p=prob).

    Формула:
        P(X = k) = C(n, k) * p^k * (1 - p)^(n - k)
    де:
        n = quantity  — кількість випробувань (подач),
        p = prob      — ймовірність «успіху» в одному випробуванні (ймовірність ейсу),
        k = likely    — рівно стільки успіхів (ейси), скільки нас цікавить.
    """
    return comb(quantity, likely) * (prob ** likely) * ((1 - prob) ** (quantity - likely))

# 15 подач (n=15), ймовірність ейсу 0.18 (p=0.18), шукаємо ймовірність рівно 4 ейсів (k=4)
p = binom(15, 0.18, 4) * 100   # переводимо з частки у відсотки

# Виводимо результат з одним знаком після коми у форматі: "Outcome: XX.X %"
print(f"Outcome: {p:.1f} %")

Outcome: 16.2 %


**Підсумкова інтерпретація (біноміальна ймовірність):**  
- При `n = 15` подачах і шансі ейса `p = 0.18`, ймовірність отримати **рівно `k = 4` ейси**:
  
- `P(X=4) = 0.161501` → **16.15%**.  
- Це означає: якщо такі умови повторювати багато разів, то **приблизно у 16.15% випадків** ми побачимо **рівно 4 ейси** з 15 подач.  
- Якщо потрібно інші `k`, формула залишається тією самою: `C(n,k)*p^k*(1-p)^(n-k)`.